# Linear models
Linear models are one of the most simple ML methods intended for regression and classification tasks, in which target value is expected to be a linear combination of features. For instance, the single variable case represents a first degree polynomial:

$$
    y(x) = mx + b
$$

where $b$ is the intercept and $m$ is the slope of the straight. In this post I want to share you what linear models exists, how are used, and how model training is performed for each one.

## Linear regression
Linear regression is the most basic approach for a regression task, here we want to find linear model that fits a continuos dependant variable $y$. In this case we are considering that our model has more than 1 independent variable, so our input, independent coefficient, slope and dependant variable will be vectors:

$$
    y^{*}(\vec{x}) = \vec{w} \cdot \vec{x} + b
$$

it's the same definition but instead of scalars we have vectors, notice that we renamed the slope coefficient $m$ to $\vec{w}$ vector that represents the weights for each features telling us how much they contribute to the model output.

### How training is performed?
The training stage consists on find the vectors $\vec{w}$ and the intercept $b$ that fits the target variable $\vec{y}$, the default approach is least squares method. Ordinary least squares is an analytical approach to find the vectors that minimizes the quadratic error function for the model outcomes. The scalar definition is given as follows:
$$
    J(\vec{y^{*}}) = \sum_{i=1}^{n} (\vec{y}_i - \vec{y^*}_i)^2
$$

where:
- $n$ is the number of samples.
- $y_i$ is the output for sample $i$.
- $\vec{y^{*}}$ is the vector containing the outputs of the model.

Square error function measures how far are the model outomes to the true sample value, the difference is squared to avoid term cancellation among errors of different inputs, resulting in $J(\vec{y^{*}}) = 0$ and then interpreted to "hey, we have the perfect fit, we don't need to find a better solution".

#### Ordinary least squares
This is a numerical approach to find an analytical solution for finding a vector $\vec{w}$ that minimizes $J(\vec{y^{*}})$ error function, let's start re-writting the loss function in terms of $w_i$ coefficients:

$$
  J(\vec{y^{*}}) = \sum_{i=1}^{n} (\vec{y}_i - b - \sum_{j=1}^{p} \vec{w_i} \vec{x_{ij}})^2
$$

The procedure is the following:

1. Differrentiate $J(\vec{y^{*}})$ error function by $\vec{w}$ components.

$$
    \frac{\partial J}{\partial w_j} = (-2\sum_{i=1}^{n} x_{ij})(\sum_{i=1}^n \vec{y}_i - b - \sum_{k=1}^{p} \vec{x}_{ik} \vec{w}_{k})
$$

2. We need to find the minimum so we equalize the  the partial derivate to 0.

$$
    0 = (-2\sum_{i=1}^{n} x_{ij})(\sum_{i=1}^n \vec{y}_i - b - \sum_{k=1}^{p} \vec{x}_{ik} \vec{w}_{k})
$$

3. Building up the product and reordering.

$$
    b \sum_{i=1}^{n} \vec{x}_{ij} + \sum_{k=1}^{p} \vec{w}_{k} \sum_{i=1}^{n} \vec{x}_{ij} \vec{x}_{ik} = \sum_{i=1}^{n} \vec{x}_{ij} \vec{y}_{i}
$$

We got a linear system of $p+1$ equations, one for the intercept and other ones for each variable weight. This approach give us the exact solution if exists, however the computations are expensive because at high level we are performing matrix operations like calculating inverse matrixes.

#### Stochastic descent gradient (SGD)
Analytical solution is fine but takes many computational resources, so a more efficient approach consists on using SGD method defining an initial solution and iterating to reduce the error. The squared error function is differentiable, let's update the vectorial definition of the model

$$
    y_{i}^{*}(\vec{x}) = \vec{w} \cdot \vec{x}
$$

having:
- $\vec{w}_0 = b$ to calculate the intercept coefficient.
- $\vec{x}_0 = 1$, represents the intercept existence.
- $||\vec{w}|| = ||\vec{x}|| = p + 1$

as we seen in [neural networks guide](https://www.kaggle.com/code/criser2013/neural-networks) we use the negative gradient of the loss function to find the direction where it comes minimal. The procedure is similar, wee need to differentiate the loss function by the model output, choose an initial solution (often a zero or random vector), select a learning rate and start iterating. The learning rule is defined as follows:

$$
    w^{(i+1)} = w^{(i)} - \eta \frac{\partial J}{\partial w}
$$

and we can add regularization coefficients or early stopping to avoid overfiting or performance downgrade. This method has a complexity of $O(kn \bar{p})$ where $k$ is the number of training epochs, $n$ the number of samples and $\bar{p}$ the average number of non-zero attributes per sample, so it has a linear cost.

### Assumptions
- Variables $x$ and $y$ have a linear relationship.
- The residuals are independent of each other, their covariance is 0.
- The residuals has a constant variance.
- The residuals follows a Gaussian distribution.

### Limitations
- Sensitive to outliners
- Assumes that the chosen model appropriately represents the relationship between the variables.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = LinearRegression(n_jobs=-1, tol=1e-7, fit_intercept=True)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

### Variations

#### Lasso
Lasso is a linear model that estimates sparse coefficients the key feature is the ability to set some coefficients to zero. It consists on adding the norm $||\vec{w}||$ to the squared error function and a strenght penalty coefficient $\lambda$.

$$
  J(\vec{y^{*}}) = \frac{1}{2n} \sum_{i=1}^{n} (\vec{y}_i - \vec{y^*}_i)^2 + \lambda \sum_{j=1}^{p} \vec{w}_{j}
$$

Promotes sparsity within the model to avoid issues of multicollinearity and overfitting but also improving model interpretability because just significant features will remain. High $\lambda$ values decreases model variance but add some bias, low values does the opposite, and a zero value transforms the loss function to the regular squared error function. Lasso is also known as **L1 regularization** and is used in other ML approaches like neural nets or SVMs to improve model generalization capabilities.

#### Ridge
The Ridge variant, also known as **L2 penalization** is an improved version of Lasso that avoids setting some weights to zero. Has the same goal, reduce overfitting and multicollinearity issues. Also avoid issues when some $w_i$ weights comes too big. To solve this situation, we add the squared norm of the weights vector $||\vec{w}||^{2}$ and a strenght term $\lambda$ to the regularization coefficient that penalizes big weights.

$$
  J(\vec{y^{*}}) = \sum_{i=1}^{n} (\vec{y}_i - \vec{y^*}_i)^2 + \lambda \sum_{j=1}^{p} \vec{w}_{j}^{2}
$$

It's similar to Lasso but it can't set weights to zero and as Lasso is a widely regularization statregy used in other ML algorithms to reduce variance introducing some bias.

#### Elastic-Net
Elastic-net is a linear model that combines Lasso and Ridge in the same loss function it has the advantages of help a sparse model with few non-zero weights while maintaining the regularization properties of Ridge regression. It works better dealing with multicollinearity than Lasso as it tends to select all correlated features together rather than randomly choosing a single feature as Lasso does.

$$
  J(\vec{y^{*}}) = \frac{1}{2n} \sum_{i=1}^{n} (\vec{y}_i - \vec{y^*}_i)^2 + \alpha \rho \sum_{j=1}^{p} \vec{w}_{j} + \frac{\alpha (1 - \rho)}{2} \sum_{j=1}^{p} \vec{w}_{j}^{2}
$$

In this case, we got $\alpha$ coefficient to control the overall strenght of the regularization and $\rho$ that determines the mix of L1 and L2 penalties. A value of 0 corresponds to Ridge regression, 1 to Lasso, and values in between to a mix of both.


The best approach to determine the optimal values for $\lambda$, $\alpha$ and $\rho$ coefficients is applying cross-validation to the model with different values.

### Implementation
Scikit-learn library implements every algorithm presented here for regression tasks at: `LinearRegression`, `Ridge`, `Lasso` and `ElasticNet` classes. Also the SGD approach for linear regression is available at `SGDRegressor` class. These approaches doesn't support multiouput tasks, so Scikit implements `MultiTask*` classes for Lasso and Elastic-net that trains as many models as required outputs.

## Logistic regression
Logistic regression is like a linear regression model but is aimed to work with classification tasks, model definition is similar but in this case the goal is to fit a categorical target function, so we need to add another component to the model because our current definition can't guarantee a categorical output because the linear combination of inputs, weights and bias returns a continuous value always. We rename the linear model to $z$ that represents the log-odds (logits) the linear output of the model

$$
    z(\vec{x}) = \vec{w} \cdot \vec{x} + b
$$

The linear output represents a score but it raw value can't be used to classify a sample, so we need to transform this score into a probability distribution. To achieve this we apply an output function to $z$ to estimate the probability of a sample to belong at certain class, depending whether is a binary classification task or a multiclass task we use the sigmoid or softmax function. To avoid repeating myself, I let you the definition of each function given at [neural networks guide](https://www.kaggle.com/code/criser2013/neural-networks) with some changes.

> - **Sigmoid:** For binary class tasks, it takes the logits of the linear combination. The sigmoid function converts the raw score into the probability of the positive class:
> $$
    p(y = 1 \mid x) = \frac{1}{1+e^{z}}
$$
>
> A class prediction can then be obtained using a threshold, commonly $0.5$:
>
> $$
    y^{*}(\vec{x}) =
    \begin{cases} 
        1 & \text{if } x \geq 0.5 \\ 
        0 & \text{otherwise}
    \end{cases}
$$
>
> - **Softmax ($\theta$):** In multiclass tasks (assuming $K$ classes) converts raw logits into a probability distribution.
>
> $$
    p(y = k \mid x) = \frac{e^{z_k}}{\sum_{i=1}^{K} e^{z_i}}
$$
>
> The resulting probabilities sum to one and the predicted class is then the class with the highest probability:
>
> $$
    y^{*}(\vec{x}) = \operatorname*{arg\,max}_{k \in \{1,\ldots, K \}}
 p(y = k \mid x)
$$

In the same way, the loss function is modified depending on the classification task, available loss functions are:

- **Binary cross entropy log-loss:** On binary tasks measures the distance between true labels and the predicted probabilities. Commonly is averaged to compute a global score for each evaluated sample.
$$
    L(\vec{y}, \vec{y^{*}}) = -\frac{1}{n} \sum_{i=1}^{n} y_{i} log(y_{i}^{*}) + (1 - y_{i}) log(1 - y_{i}^{*})
$$

- **Categorical cross entropy log-loss:** For multiclass tasks, compares the model's probability distribution with the true label.
$$
  L(\vec{y}, \vec{y^{*}}) =  - \sum_{i=1}^{C} y_{i} log(y_{i}^{*})
$$

The training stage uses the **SGD** approach to find the minimal error direction, there isn't a closed formula to find an analytical solution easily because we have apply sigmoid or softmax function to obtain a probability distribution. Even as regular linear regression we can add regularization coefficients as Ridge, Lasso or Elastic-net in the loss function to avoid model overfitting.

### Implementation
Scikit-learn has the `LogisticRegression` class to implement a logistic regression model, by default it adds some regularization to the model.

In [ ]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = MLPClassifier(hidden_layer_sizes=(20, 15, 10, 5), solver="lbfgs", activation="relu", random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

## Most common hyperparameters

## Advantages
- a

## Disadvantages
- a